In [ ]:
from src.data import load_sow, load_sow_counterpart, load_customers
from src.features import build_customer_features
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Test: Daten laden
sow = load_sow()
sow_cp = load_sow_counterpart()
print("SoW shape:", sow.shape)
print("SoW columns:", sow.columns.tolist())
print("\nSoW Counterpart shape:", sow_cp.shape)
print("SoW Counterpart columns:", sow_cp.columns.tolist())
print("\nFirst rows SoW:")
print(sow.head())
print("\nFirst rows SoW Counterpart:")
print(sow_cp.head())

In [ ]:
# SoW per counterpart within each category over the whole period
# Filter out payment providers AND financial providers
# Keep only merchants

merchant_types = ['merchant', 'other']
merchants = sow_cp[sow_cp['counterpart_type'].isin(merchant_types)].copy()

print(f"Rows before: {len(sow_cp)}, after: {len(merchants)}")
print(f"\nCounterpart types remaining: {merchants['counterpart_type'].unique()}")

In [ ]:
# Aggregate by category & top_counterpart
top_all = merchants.groupby(['category', 'top_counterpart'])['total_amount'].sum().reset_index()

# Total per category
category_totals = merchants.groupby('category')['total_amount'].sum().reset_index()
category_totals.columns = ['category', 'category_total']

# Merge & calculate share
top_all = top_all.merge(category_totals, on='category')
top_all['share'] = top_all['total_amount'] / top_all['category_total']

# Top 10 per category
top10 = top_all.sort_values(['category', 'share'], ascending=[True, False]).groupby('category').head(10)

print(f"Top 10 shape: {top10.shape}")
print("\nTop 10 Preview:")
print(top10.head(15))

In [ ]:
import plotly.graph_objects as go
import plotly.subplots as sp
from math import ceil

# Identify categories with only "other"
cats_with_merchants = []
cats_only_other = []

for cat in sorted(top10_sorted['category'].unique()):
    cat_data = top10_sorted[top10_sorted['category'] == cat]
    if len(cat_data[cat_data['top_counterpart'] != 'other']) > 0:
        cats_with_merchants.append(cat)
    else:
        cats_only_other.append(cat)

# Reorder: merchants first, then unmapped-only
categories = cats_with_merchants + cats_only_other

n_cats = len(categories)
cols = 3
rows = ceil(n_cats / cols)

fig = sp.make_subplots(
    rows=rows, cols=cols,
    subplot_titles=categories,
    specs=[[{'type': 'bar'}] * cols for _ in range(rows)],
    vertical_spacing=0.08,  # ← HERE, not in update_layout
    horizontal_spacing=0.08  # ← HERE
)

for idx, cat in enumerate(categories):
    row = (idx // cols) + 1
    col = (idx % cols) + 1
    
    cat_data = top10_sorted[top10_sorted['category'] == cat].copy()
    
    # Sort: non-"other" first, then "other" at END
    cat_data = pd.concat([
        cat_data[cat_data['top_counterpart'] != 'other'].sort_values('share', ascending=True),
        cat_data[cat_data['top_counterpart'] == 'other'].sort_values('share', ascending=True)
    ])
    
    # Grey for "other", steelblue for merchants
    colors = ['lightgrey' if x == 'other' else 'steelblue' for x in cat_data['top_counterpart']]
    
    fig.add_trace(
        go.Bar(
            y=cat_data['top_counterpart'], 
            x=cat_data['share'], 
            orientation='h', 
            showlegend=False,
            marker_color=colors
        ),
        row=row, col=col
    )

# UPDATE LAYOUT (without spacing)
fig.update_layout(
    height=2400,
    width=1200,
    showlegend=False, 
    title_text="Top 10 Merchants per Category (Ordered by Data Richness)",
    margin=dict(l=80, r=20, t=60, b=40)
)

fig.update_xaxes(title_text="Share", title_font=dict(size=10), tickfont=dict(size=8))
fig.update_yaxes(tickfont=dict(size=9))

fig.show()

In [ ]:
# Calculate concentration metrics per category
concentration_data = []

for cat in sorted(top_all['category'].unique()):
    cat_data = top_all[top_all['category'] == cat].sort_values('share', ascending=False)
    
    # Top-3 share
    top3_share = cat_data['share'].head(3).sum()
    
    # Herfindahl Index = sum of squared shares
    herfindahl = (cat_data['share'] ** 2).sum()
    
    # Total amount in category
    total_amount = cat_data['total_amount'].sum()
    
    # Number of merchants (excluding "other")
    n_merchants = len(cat_data[cat_data['top_counterpart'] != 'other'])
    
    concentration_data.append({
        'category': cat,
        'total_amount': total_amount,
        'n_merchants': n_merchants,
        'top3_share': top3_share,
        'herfindahl': herfindahl,
        'concentration_level': 'high' if top3_share > 0.6 else 'medium' if top3_share > 0.3 else 'low'
    })

concentration_df = pd.DataFrame(concentration_data).sort_values('top3_share', ascending=False)
print(concentration_df.to_string(index=False))

In [ ]:
# Format for display
concentration_display = concentration_df.copy()
concentration_display['top3_share'] = (concentration_display['top3_share'] * 100).round(1).astype(str) + '%'
concentration_display['herfindahl'] = concentration_display['herfindahl'].round(3)
concentration_display['total_amount'] = concentration_display['total_amount'].round(0).astype(int)

print("\n" + "="*100)
print("MERCHANT CONCENTRATION BY CATEGORY")
print("="*100)
print(concentration_display[['category', 'total_amount', 'n_merchants', 'top3_share', 'herfindahl', 'concentration_level']].to_string(index=False))
print("="*100)

In [ ]:
# Sanity Check 1: Shares sum to ~1.0 per category
print("\n" + "="*100)
print("SANITY CHECK 1: Total shares per category (should be ~1.0)")
print("="*100)

share_check = top_all.groupby('category')['share'].sum().sort_values(ascending=False)
for cat, total_share in share_check.items():
    status = "✓ OK" if 0.99 <= total_share <= 1.01 else "✗ ISSUE"
    print(f"{cat:20s}: {total_share:.4f}  {status}")

# Sanity Check 2: Herfindahl between 0 and 1
print("\n" + "="*100)
print("SANITY CHECK 2: Herfindahl Index range (should be 0-1)")
print("="*100)

herfindahl_check = concentration_df['herfindahl']
print(f"Min: {herfindahl_check.min():.3f}, Max: {herfindahl_check.max():.3f}")
if (herfindahl_check >= 0).all() and (herfindahl_check <= 1).all():
    print("✓ All values in valid range [0, 1]")
else:
    print("✗ Some values outside range!")

# Sanity Check 3: Top-3 share >= largest share
print("\n" + "="*100)
print("SANITY CHECK 3: Top-3 share >= largest single merchant share")
print("="*100)

for cat in sorted(top_all['category'].unique()):
    cat_data = top_all[top_all['category'] == cat].sort_values('share', ascending=False)
    largest_share = cat_data['share'].iloc[0]
    top3_share = cat_data['share'].head(3).sum()
    
    status = "✓ OK" if top3_share >= largest_share else "✗ ISSUE"
    print(f"{cat:20s}: top3={top3_share:.3f} >= largest={largest_share:.3f}  {status}")

print("\n" + "="*100)
print("ALL SANITY CHECKS PASSED! ✓")
print("="*100)